First lets import the trading client so we can interact w/ platform

In [1]:
import aiohttp
import asyncio

import sys
import os

# Add parent directory to path
sys.path.append(os.path.abspath('trading-simulator-client'))

from trading_client import *

Next lets import our utils and configuration variables of course

In [1]:
from utils import compute_ev_df
from config import game_id, token

Now for some basic helper functions to be used in potential strategies

In [2]:
# strategy helperss

import pandas as pd

# generate signals
def get_signal(edge):
    if edge > 0:
        return "LONG"
    elif edge < 0:
        return "SHORT"
    else:
        return "HOLD"

    # adjust fair value
def adjust_fv(row, positions):
    if row["team"] in positions:
        pos = positions[row["team"]]
    else:
        pos = 0
    return row["fair_value"] - LAMBDA * pos


# final trading decision
def execution_decision(row):
    price = row["market_price"]
    
    if price < row["adj_buy_below"]:
        return "BUY"
    elif price > row["adj_sell_above"]:
        return "SELL"
    else:
        return "NO TRADE"
    
# detect adverse move relative to edge
def is_adverse(row):
    if pd.isna(row["prev_price"]):
        return False
    if row["edge"] > 0:
        # want to go LONG → bad if price is rising
        return row["price_change"] > 0
    else:
        # want to go SHORT → bad if price is falling
        return row["price_change"] < 0
    
# compute trade size
def compute_size(row, positions):
    """
    Compute trade size dynamically:
    - Scales with edge
    - Penalizes if market moves against you (reversal)
    - Limits exposure based on current position
    """

    # Scale size by edge
    edge_strength = min(abs(row["edge"]) / EDGE_SCALE, 1.0)
    size = BASE_SIZE * edge_strength

    # Reduce size if market moved against your intended trade
    if row["adverse_move"]:
        size *= REVERSAL_PENALTY

    # Adjust for current position
    current_pos = positions.get(row["team"], 0)
    if row["edge"] > 0:
        # Want to go LONG
        available = MAX_POSITION - current_pos
    else:
        # Want to go SHORT
        available = MAX_POSITION + current_pos  # current_pos is negative if short

    # Don't exceed remaining capacity
    size = min(size, max(available, 0))

    return int(size)

# compute marked mid prices
def mid_price_map(order_books, fair_values):
    '''
    only compute mid price maps for relevant teams
    '''
    hashmap = {}
    for key in fair_values:
        book = order_books[key]
        hashmap[key] = (book.best_bid_px + book.best_ask_px) / 2
    return hashmap


Lets also have some trade execution logic helpers, specifically written to interface with API and its components

In [4]:
# execution helpers

# get open orders to avoid duplicates
def has_open_orders(team, action, price, open_orders):
    for order in open_orders.values():
        if order.display_symbol != team:
            continue

        side = "BUY" if order.qty > 0 else "SELL"

        # Match side + price
        if side == action and abs(order.px - price) < PRICE_TOL:
            return True
    return False

First lets try and liquidity taking edge based strategy which buys and sells according to mis-priced markets

In [5]:
# this isnt really market making

# it is trading based on perceived edge

# not a bad strategy but not what im going for

LAMBDA = 0.02  # inventory penalty

EDGE_THRESHOLD = 2.0
EDGE_PCT_THRESHOLD = 0.15

MAX_POSITION = 60
BASE_SIZE = 30          # Base units per trade
EDGE_SCALE = 10         # Scale edge to 0-1 for sizing
REVERSAL_PENALTY = 0.3  # Reduce size if market moves against you

PRICE_TOL = 1  # only place order again if new order is this much more (or less) than open order

class TradingBot(Client):
    def __init__(
        self,
        session: aiohttp.ClientSession,
        game_id: int,
        token: str,
    ) -> None:
        super().__init__(session, game_id, token)

    async def on_start(self) -> None:
        prev_prices = {}  # will store last tick's market price
        
        while True:
            
            order_books = await self.get_order_books()
            positions = self.positions
            
            # trading logic 
            fair_value_df = compute_ev_df()
            fair_values = dict(zip(fair_value_df["team"], fair_value_df["EV"]))
            
            market_prices = mid_price_map(order_books)
            
            common_teams = set(fair_values) & set(market_prices)

            df = pd.DataFrame({
                "team": list(common_teams),
            })

            df["fair_value"] = df["team"].map(fair_values)
            df["market_price"] = df["team"].map(market_prices)

            # --- Step 2: Compute edge ---
            df["edge"] = df["fair_value"] - df["market_price"]
            df["edge_pct"] = df["edge"] / df["market_price"]

            # --- Step 3: Filter meaningful trades ---


            df["trade_candidate"] = (
                (df["edge"].abs() > EDGE_THRESHOLD) |
                (df["edge_pct"].abs() > EDGE_PCT_THRESHOLD)
            )

            df["signal"] = df["edge"].apply(get_signal)

            # --- Step 5: Define trading band ---
            DELTA = 1.5  # spread buffer

            df["adj_fair_value"] = df.apply(adjust_fv, axis=1, args=(positions,))

            df["adj_buy_below"] = df["adj_fair_value"] - DELTA
            df["adj_sell_above"] = df["adj_fair_value"] + DELTA

            df["action"] = df.apply(execution_decision, axis=1)

            # --- Step 8: Rank trades ---
            df = df.sort_values(by="edge", ascending=False)
            
            # Track previous prices (initially use current market as starting point)
            df["prev_price"] = df["team"].map(lambda t: prev_prices.get(t, df.loc[df["team"] == t, "market_price"].values[0]))
            df["prev_price"] = df["team"].map(prev_prices)

            # Compute price change
            df["price_change"] = df["market_price"] - df["prev_price"]

            # compute adverse moves
            df["adverse_move"] = df.apply(is_adverse, axis=1)

            # insert size
            df["size"] = df.apply(lambda row: compute_size(row, positions), axis=1)

            # Update prev_prices after each tick for next iteration
            for team, price in zip(df["team"], df["market_price"]):
                prev_prices[team] = price

            # filter out all no-trades for easy viewing
            tradeable_df = df.loc[(df.action != "NO TRADE") & (df.trade_candidate == True)]   
            
            # get current open orders
            current_open_orders = await self.get_open_orders()
            
            # --- Trading Logic ---
            for _, row in tradeable_df.iterrows():
                team = row["team"]
                size = row["size"]
                price = row["market_price"]
                action = row["action"]
                
                # skip if already have order open at that price for this action
                if has_open_orders(team, action, price, current_open_orders):
                    continue

                if size <= 0:
                    continue  # Skip if no size to trade

                # Determine order type and quantity
                if action == "BUY":
                    order_type = "LIMIT"
                    qty = size  # positive quantity for buy
                elif action == "SELL":
                    order_type = "LIMIT"
                    qty = -size  # negative quantity for sell
                else:
                    continue  # just in case

                try:
                    open_order = await self.send_order(team, price, qty, order_type)
                    print(
                        f"Placed {action:4s} order | Team: {team:12s} | "
                        f"Qty: {qty:3d} | Price: {price:.2f} | Edge: {row['edge']:.2f}"
                    )
                except Exception as e:
                    print(f"Failed to place order for {team}: {e}")
            
            # now fetch and cancel all stale open orders
            current_open_orders = await self.get_open_orders()

            # Cancel any stale orders
            for order_id, order in current_open_orders.items():
                team = order.display_symbol
                side = "BUY" if order.qty > 0 else "SELL"

                # Determine stale condition
                # Example: edge no longer meets threshold
                if team in fair_values:
                    edge = df.loc[df["team"] == team, "edge"].values[0]
                    if (side == "BUY" and edge < EDGE_THRESHOLD) or (side == "SELL" and edge > -EDGE_THRESHOLD):
                        try:
                            await self.cancel_orders([order_id])
                            print(f"Cancelled stale {side} order for {team} at {order.px}")
                        except Exception as e:
                            print(f"Failed to cancel stale order {order_id} for {team}: {e}")
                    
            print("\n")

            await asyncio.sleep(60)

async def main():
    async with create_session() as session:
        client = TradingBot(session, game_id, token)
        print(f"Access web view at {client.web_url}")
        await client.start()


await main()

Access web view at https://games.drw.com/games/trading-simulator/160
Placed BUY  order | Team: Tennessee    | Qty:   8 | Price: 10.39 | Edge: 4.22
Placed BUY  order | Team: UConn        | Qty:   6 | Price: 12.19 | Edge: 3.49
Placed BUY  order | Team: Duke         | Qty:   4 | Price: 26.99 | Edge: 2.25


-------- CANCELING ORDERS --------
Cancelled BUY order for Tennessee at 10.39
Cancelled BUY order for UConn at 12.19
Cancelled BUY order for Duke at 26.99


CancelledError: 

That is nice, but Im more interested in participating from a liquidity provider position.

So lets try that

In [5]:
# market making here

# bid and ask globals
ALPHA = 0.8             # between [0.5, 0.9] supposedly is best
LAMBDA = 0.02           # inventory penalty
DELTA = 2               # spread buffer

# edge threshold globals
# EDGE_THRESHOLD = 2.0
# EDGE_PCT_THRESHOLD = 0.15

EDGE_THRESHOLD = 1
EDGE_PCT_THRESHOLD = 0.10

# position sizing globals
MAX_POSITION = 60
BASE_SIZE = 20          # Base units per trade
EDGE_SCALE = 10         # Scale edge to 0-1 for sizing
REVERSAL_PENALTY = 0.3  # Reduce size if market moves against you

TICK = 0.5              # minimum tick size

# threshold for cancel
# and replace related to valuation
VALUATION_THRESHOLD = 1


# logger configuration
import logging
import sys

# Configure logger
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# Prevent duplicate handlers
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter('%(levelname)s - %(message)s'))
    logger.addHandler(handler)

# this will disable output (comment out for debugging)
logging.disable(logging.CRITICAL)


class TradingBot(Client):
    def __init__(
        self,
        session: aiohttp.ClientSession,
        game_id: int,
        token: str,
    ) -> None:
        super().__init__(session, game_id, token)

    async def on_start(self) -> None:
        prev_prices = {}  # will store last iterations market price
        
        while True:
            
            order_books = await self.get_order_books()
            positions = self.positions
            
            # fair value logic
            fair_value_df = compute_ev_df()
            fair_values = dict(zip(fair_value_df["team"], fair_value_df["EV"]))
            
            market_prices = mid_price_map(order_books, fair_values)
            
            common_teams = set(fair_values) & set(market_prices)

            df = pd.DataFrame({
                "team": list(common_teams),
            })

            df["fair_value"] = df["team"].map(fair_values)
            df["market_price"] = df["team"].map(market_prices)

            # edge computation
            df["edge"] = df["fair_value"] - df["market_price"]
            df["edge_pct"] = df["edge"] / df["market_price"]

            
            # filtering for trades w/ larger edge
            df["trade_candidate"] = (
                (df["edge"].abs() > EDGE_THRESHOLD) |
                (df["edge_pct"].abs() > EDGE_PCT_THRESHOLD)
            )
            
            # Track previous prices (initially use current market as starting point)
            # for sizing
            df["prev_price"] = df["team"].map(lambda t: prev_prices.get(t, df.loc[df["team"] == t, "market_price"].values[0]))
            df["prev_price"] = df["team"].map(prev_prices)

            # Compute price change
            df["price_change"] = df["market_price"] - df["prev_price"]

            # compute adverse moves
            df["adverse_move"] = df.apply(is_adverse, axis=1)

            # insert size
            df["size"] = df.apply(lambda row: compute_size(row, positions), axis=1)
            
            # Update prev_prices after each tick for next iteration
            for team, price in zip(df["team"], df["market_price"]):
                prev_prices[team] = price
            
            # I like this idea of only making markets where
            # perceived edge is higher
            # so from here down is where the logic comes in
            
            # segragating portfolio into net longs and shorts
            net_shorts = df.loc[(df.edge < 0) & (df.trade_candidate)].copy(deep=True)
            net_longs = df.loc[(df.edge > 0) & (df.trade_candidate)].copy(deep=True)
            
            
            # quote based on net long or short
            def inventory_adjustment(quote, team, positions):
                '''
                ie factoring in skew
                
                this is analagous to binary search, move your
                pointers around to account from people lifting 
                offers and hitting bids
                '''
                if team in positions:
                    return quote - (LAMBDA * positions[team])
                else:
                    return quote

            # longs first
            net_longs["mid_quote"] = net_longs["fair_value"] - ALPHA * (net_longs["fair_value"] - net_longs["market_price"])
            net_longs["bid_quote"] = net_longs["mid_quote"] - (DELTA / 2)
            net_longs["ask_quote"] = net_longs["mid_quote"] + (DELTA / 2)
            net_longs["bid_adj"] = net_longs.apply(
                    lambda row: inventory_adjustment(row["bid_quote"], row["team"], positions),
                    axis=1
            )
            net_longs["ask_adj"] = net_longs.apply(
                    lambda row: inventory_adjustment(row["ask_quote"], row["team"], positions),
                    axis=1
            )
            
            
            # next shorts
            net_shorts["mid_quote"] = net_shorts["fair_value"] + ALPHA * (net_shorts["market_price"] - net_shorts["fair_value"])
            net_shorts["bid_quote"] = net_shorts["mid_quote"] - (DELTA / 2)
            net_shorts["ask_quote"] = net_shorts["mid_quote"] + (DELTA / 2)
            net_shorts["bid_adj"] = net_shorts.apply(
                    lambda row: inventory_adjustment(row["bid_quote"], row["team"], positions),
                    axis=1
            )
            net_shorts["ask_adj"] = net_shorts.apply(
                    lambda row: inventory_adjustment(row["ask_quote"], row["team"], positions),
                    axis=1
            )
                       

            # before we send out new quotes, fetch and cancel all open orders
            # where markets have moved by at least TICK
            # As well as resting orders which no longer aggree with our valuations
            # this can be done by seeing if its in net shorts or net longs
            # we dont want to be trading in those markets anyways
            
            # cancel & replace by tick (ie if abs(prev - market_price) >= TICK)
            cancel_teams = set(df.loc[(abs(df.market_price - df.prev_price) >= TICK)].team)
            stale_val_teams = set(df.team) - (set(net_longs.team) | set(net_shorts.team))
            
            logger.info("-------- CANCELING ORDERS --------")
            current_open_orders = await self.get_open_orders()
            for order_id, order in current_open_orders.items():
                team = order.display_symbol
                if (team in cancel_teams) or (team in stale_val_teams): # only cancel if new tick or stale val
                    side = "BUY" if order.qty > 0 else "SELL"
                    try:
                        await self.cancel_orders([order_id])
                        logger.info(f"Cancelled {side} order for {team} at {order.px}")
                    except Exception as e:
                        logger.info(f"Failed to cancel order {order_id} for {team}: {e}")
                        
            # this is goood logic, but it is incomplete. It adjusts quotes based on 
            # changes in market information, but it should also consider changes
            # in valuations
            
            # this will be done when considering sending out new bids
            # so, to do that, we need to construct our buying and selling
            # maps for easier reference later
            buy_orders, sell_orders = {}, {}
            current_open_orders = await self.get_open_orders()
            for order_id, order in current_open_orders.items():
                team = order.display_symbol
                side = "BUY" if order.qty > 0 else "SELL"
                if side == "BUY":
                    if team in buy_orders:
                        raise Exception(f"have more than one buy order for team {team}")
                    else:
                        # already have objects w all necessary info
                        buy_orders[team] = {"order_id": order_id, "order_obj": order}
                elif side == "SELL":
                    if team in sell_orders:
                        raise Exception(f"have more than one sell order for team {team}")
                    else:
                        # already have objects w all necessary info
                        sell_orders[team] = {"order_id": order_id, "order_obj": order}
            
            
            logger.info("------- SENDING NEW ORDERS -------")
            
            order_type = "LIMIT"
            
            # net longs first
            for _, row in net_longs.iterrows():
                team = row["team"]
                size = row["size"]
                bid, ask = row["bid_adj"], row["ask_adj"]
                market_price = row["market_price"]

                if size <= 0:
                    continue  # Skip if no size to trade
                    
                
                # place buying order
                try:
                    # only place if we either dont already have one outstanding
                    # or our valuation has changed
                    if team not in buy_orders:
                        open_order = await self.send_order(team, bid, size, order_type)
                        logger.info(
                            f"Placed OPENING BUY order | Team: {team:12s} | "
                            f"Qty: {size:3d} | Market Price: {market_price:.2f} | Price: {bid:.2f} | Edge: {row['edge']:.2f}"
                        )
                    else:
                        order_id, order = buy_orders[team]["order_id"], buy_orders[team]["order_obj"]
                        # only cancel & replace if valuation has changed
                        if abs(order.px - bid) >= VALUATION_THRESHOLD:
                            # cancel
                            try:
                                await self.cancel_orders([order_id])
                                logger.info(f"Cancelled {side} order for {team} at {order.px}")
                            except Exception as e:
                                logger.info(f"Failed to cancel order {order_id} for {team}: {e}")
                            
                            # replace
                            open_order = await self.send_order(team, bid, size, order_type)
                            logger.info(
                                f"Placed Replacement BUY order | Team: {team:12s} | "
                                f"Qty: {size:3d} | Market Price: {market_price:.2f} | Price: {bid:.2f} | Edge: {row['edge']:.2f}"
                            )
                            
                except Exception as e:
                    print(f"Failed to update BID net long quote for {team}: {e}")

                # selling orders
                try:
                    # only place if we either dont already have one outstanding
                    # or our valuation has changed
                    if team not in sell_orders:
                        open_order = await self.send_order(team, ask, -1 * size, order_type)
                        logger.info(
                            f"Placed OPENING SELL order | Team: {team:12s} | "
                            f"Qty: {size:3d} | Market Price: {market_price:.2f} | Price: {ask:.2f} | Edge: {row['edge']:.2f}"
                        )
                    else:
                        order_id, order = sell_orders[team]["order_id"], sell_orders[team]["order_obj"]
                        # only cancel & replace if valuation has changed
                        if abs(order.px - ask) >= VALUATION_THRESHOLD:
                            # cancel 
                            try:
                                await self.cancel_orders([order_id])
                                logger.info(f"Cancelled {side} order for {team} at {order.px}")
                            except Exception as e:
                                logger.info(f"Failed to cancel order {order_id} for {team}: {e}")
                            
                            # replace
                            open_order = await self.send_order(team, ask, -1 * size, order_type)
                            logger.info(
                                f"Placed Replacement SELL order | Team: {team:12s} | "
                                f"Qty: {size:3d} | Market Price: {market_price:.2f} | Price: {ask:.2f} | Edge: {row['edge']:.2f}"
                            )
   
                except Exception as e:
                    logger.info(f"Failed to update ASK net long quote for {team}: {e}")

            # next up is the net shorts
            for _, row in net_shorts.iterrows():
                team = row["team"]
                size = row["size"]
                bid, ask = row["bid_adj"], row["ask_adj"]

                if size <= 0:
                    continue  # Skip if no size to trade
                    
                # place buying order
                try:
                    # only place if we either dont already have one outstanding
                    # OR our valuation has changed
                    if team not in buy_orders:
                        open_order = await self.send_order(team, bid, size, order_type)
                        logger.info(
                            f"Placed OPENING BUY order | Team: {team:12s} | "
                            f"Qty: {size:3d} | Market Price: {market_price:.2f} | Price: {bid:.2f} | Edge: {row['edge']:.2f}"
                        )
                    else:
                        order_id, order = buy_orders[team]["order_id"], buy_orders[team]["order_obj"]
                        # only cancel & replace if valuation has changed
                        if abs(order.px - bid) >= VALUATION_THRESHOLD:
                            # cancel
                            try:
                                await self.cancel_orders([order_id])
                                logger.info(f"Cancelled {side} order for {team} at {order.px}")
                            except Exception as e:
                                logger.info(f"Failed to cancel order {order_id} for {team}: {e}")
                            
                            # replace
                            open_order = await self.send_order(team, bid, size, order_type)
                            logger.info(
                                f"Placed BUY order | Team: {team:12s} | "
                                f"Qty: {size:3d} | Market Price: {market_price:.2f} | Price: {bid:.2f} | Edge: {row['edge']:.2f}"
                            )
                            
                except Exception as e:
                    logger.info(f"Failed to update BID net short quote for {team}: {e}")

                # selling orders
                try:
                    # only place if we either dont already have one outstanding
                    # or our valuation has changed
                    if team not in sell_orders:
                        open_order = await self.send_order(team, ask, -1 * size, order_type)
                        logger.info(
                            f"Placed OPENING SELL order | Team: {team:12s} | "
                            f"Qty: {size:3d} | Market Price: {market_price:.2f} | Price: {ask:.2f} | Edge: {row['edge']:.2f}"
                        )
                    else:
                        order_id, order = sell_orders[team]["order_id"], sell_orders[team]["order_obj"]
                        # only cancel & replace if valuation has changed
                        if abs(order.px - ask) >= VALUATION_THRESHOLD:
                            # cancel 
                            try:
                                await self.cancel_orders([order_id])
                                logger.info(f"Cancelled {side} order for {team} at {order.px}")
                            except Exception as e:
                                logger.info(f"Failed to cancel order {order_id} for {team}: {e}")
                            
                            # replace
                            open_order = await self.send_order(team, ask, -1 * size, order_type)
                            logger.info(
                                f"Placed SELL order | Team: {team:12s} | "
                                f"Qty: {size:3d} | Market Price: {market_price:.2f} | Price: {ask:.2f} | Edge: {row['edge']:.2f}"
                            )
   
                except Exception as e:
                    logger.info(f"Failed to update ASK net short quote for {team}: {e}")
        
            
            # every 30 seconds execute again
            await asyncio.sleep(30)

async def main():
    async with create_session() as session:
        client = TradingBot(session, game_id, token)
        print(f"Access web view at {client.web_url}")
        await client.start()


await main()


Access web view at https://games.drw.com/games/trading-simulator/160


TypeError: unsupported operand type(s) for +: 'float' and 'NoneType'

Next steps: 


Market Making strat
1. Execution Logic
    - include logic to dump poor performing assets (is this good idea?)
3. Edge Threshold
    - should it be $2 and 15pct ? Maybe lower as tournament progresses?
    - should I decrease edge scale to increase position sizes?
4. Review trading logic

Potential Improvements

1. Quote any team so long as market price is different from expected payoff
    - potentially pick off people who aren't paying attention
2. Implement hedging
    - right now I do not hedge
3. Include order tracking so I can know which orders get filled and from there compute things such as
    - average entry price
    - lowest/highest/most recent
        - this allows us to always attempt to sell for more than we purchase for